In [ ]:

import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
from tqdm.auto import tqdm
import pickle
from dimenet_clip import (
    DimeNetPretrainer, compute_denoising_loss
) 

In [ ]:
def collate_fn(batch):
    z_pockets = []
    pocket_pos_list = []
    pocket_batch = []
    
    for i, sample in enumerate(batch):
        z = torch.tensor(sample[0], dtype=torch.int)
        z_pockets.append(z)
        pocket_pos = torch.tensor(sample[1], dtype=torch.float)
        pocket_pos_list.append(pocket_pos)
        pocket_batch.append(torch.full(z.shape, i, dtype=torch.long))
        
    return {
        'z': torch.cat(z_pockets, dim=0),
        'pos': torch.cat(pocket_pos_list, dim=0),
        'batch': torch.cat(pocket_batch, dim=0)
    }

class PocketDataset(Dataset):
    def __init__(self, pocket_pos_list, z_pocket_list):
        self.pocket_pos_list = pocket_pos_list
        self.z_pocket_list = z_pocket_list

    def __len__(self):
        return len(self.pocket_pos_list)

    def __getitem__(self, idx):
        return self.z_pocket_list[idx], self.pocket_pos_list[idx]

In [ ]:
device = 'cuda'
mol_type = 'ligand'
batch_size = 6
data_dir = '/path/to/denoising/data'  # output of data_prep/{ligand,pocket}_prep.py

In [ ]:
model = DimeNetPretrainer(
    hidden_channels=128,
    num_blocks=6
).to(device)

In [ ]:
bb_state_dict = torch.load('../weights/qm9_pretrained/backbone_U.pt')
ener_state_dict = torch.load('../weights/qm9_pretrained/readout_U.pt')
model.backbone.load_state_dict(bb_state_dict)
model.energy_readout.load_state_dict(ener_state_dict)
model.eval()

In [ ]:
with open(os.path.join(data_dir, f'h_{mol_type}_pos.pkl'), 'rb') as f:
    pos_list = pickle.load(f)
with open(os.path.join(data_dir, f'h_z_{mol_type}.pkl'), 'rb') as f:
    z_list = pickle.load(f)


train_dataset = PocketDataset(
    pos_list, z_list
)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

In [ ]:
with open(os.path.join(data_dir, f'{mol_type}_pos_valid.pkl'), 'rb') as f:
    valid_pos_list = pickle.load(f)
with open(os.path.join(data_dir, f'z_{mol_type}_valid.pkl'), 'rb') as f:
    valid_z_list = pickle.load(f)

valid_dataset = PocketDataset(
    valid_pos_list, valid_z_list
)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)


In [ ]:
def compute_denoising_loss(model, data, noise_std=0.1, device='cuda'):
    """
    1. Add noise to coordinates.
    2. Predict Energy of noisy state.
    3. Calculate Force (-Gradient of Energy).
    4. Minimize MSE(Force, Restoring_Vector).
    """
    z, pos, batch = data['z'].to(device), data['pos'].to(device), data['batch'].to(device)
    
    # Ensure gradients are enabled for force calculation (even in eval mode)
    with torch.enable_grad():
        # 1. Generate Noise
        noise = torch.randn_like(pos) * noise_std
        pos_noisy = pos + noise
        pos_noisy.requires_grad_(True) # Crucial for calculating force
        
        # 2. Forward Pass (Get Energy)
        energy = model(z, pos_noisy, batch)
        print(f"Energy: {energy.item():.4f}")
        # 3. Calculate Gradient (Force field)
        # We only need create_graph=True if we are training (to backprop through force)
        is_training = model.training
        force = -torch.autograd.grad(
            outputs=energy,
            inputs=pos_noisy,
            create_graph=is_training, 
            retain_graph=is_training
        )[0]
    
    # 4. Target: The vector that points back to the clean position
    # If pos_noisy = pos + noise, then (pos - pos_noisy) = -noise
    target_force = -noise  
    
    print(f"Force Norm: {force.norm().item():.4f}, Target Force Norm: {target_force.norm().item():.4f}")
    
    # Optional: You can scale the target by 1/sigma^2 (Score Matching theory), 
    # but for simple pre-training, matching the vector directly works well.
    
    # MSE Loss
    loss = torch.nn.functional.mse_loss(force, target_force)
    return loss

In [ ]:
train_dataset[1]

In [ ]:
torch.cuda.empty_cache()
torch.autograd.set_detect_anomaly(False)
nan_check = 0
val_loss = 0.0
tqdm_iter = tqdm(valid_loader)
for batch_data in tqdm_iter:
    loss = compute_denoising_loss(model, batch_data, noise_std=0.1, device=device)
    tqdm_iter.set_postfix_str(f"Val Loss: {loss.item():.4f}, NaN Count: {nan_check}")
    if loss.isnan().any():
        nan_check += loss.isnan().sum().item()
        continue
    val_loss += loss.item()
avg_val_loss = torch.tensor(val_loss / len(valid_loader))
print(avg_val_loss)
print(nan_check)

In [ ]:
z, pos, batch = batch_data['z'].to(device), batch_data['pos'].to(device), batch_data['batch'].to(device)
xs, rbf, i, n_nodes, batch = model.backbone(z, pos, batch)

In [ ]:
(i == 5).sum()

In [ ]:
torch.where(batch==0)

In [ ]:
(batch == 5).sum()

In [ ]:
i.shape

In [ ]:
n_nodes

In [ ]:
rbf.shape

In [ ]:
x_init.shape

In [ ]:
pos.shape

In [ ]:
x_init = xs[0]

In [ ]:
6646/514

In [ ]:
nan_check

In [ ]:
loss

In [ ]:
arr = np.ones((3,4))

In [ ]:
arr

In [ ]:
arr.sum(axis=0)

In [ ]:
from torch.nn import Linear

In [ ]:
from torch_geometric.nn.resolver import activation_resolver

In [ ]:
att_gate = Linear(128,1)

In [ ]:
arr = torch.arange(128*4, dtype=torch.float).view(4,-1)

In [ ]:
arr.shape

In [ ]:
alpha = att_gate(x_init.cpu())

In [ ]:
alpha.shape

In [ ]:
alpha

In [ ]:
alpha.sigmoid()

In [ ]:
alpha.sum(0)

In [ ]:
act = activation_resolver('swish')

In [ ]:
act(alpha)

In [ ]:
alpha.softmax(0)

In [ ]:
coords = torch.arange(0,36, dtype=torch.float32).view(-1,3)

In [ ]:
perturb_frac = 0.3
n_atoms = coords.shape[0]
n_perturb = int(np.ceil(n_atoms*perturb_frac))
randind = torch.randperm(n_atoms)[:n_perturb]

noise = torch.randn((n_perturb, 3))
coords[randind] += noise

In [ ]:
coords

In [ ]:
batch = torch.stack([torch.zeros(4,3, dtype=torch.int)+i for i in range(4)])

In [ ]:
batch